# Deploy Binary Text Segmentation Demo to HuggingFace Spaces
* Author: Juan Pablo Triana Martinez + Claude Code
* Date: 2026-04-19

This notebook builds a **self-contained** `binary_text_segmentation` demo folder ready for upload to HuggingFace Spaces.

### Folder structure produced
```
demos/
└── binary_text_segmentation/
    ├── linknet_binary_doclaynet_20_percent_seed_7_ce_0.25_dice_1.0_.pth
    ├── app.py
    ├── examples/
    │   ├── binary_1_img.png
    │   ├── binary_1_mask.png
    │   ├── binary_2_img.png
    │   ├── binary_2_mask.png
    │   ├── binary_3_img.png
    │   └── binary_3_mask.png
    ├── model.py
    └── requirements.txt
```

### Prerequisites
- Trained model `.pth` must exist in `models/`
- Example images must exist in `examples/` (run `Gradio_Binary_Text_Segmentation_demo.ipynb` first)

## 0. Setup — Imports and Paths

In [1]:
import shutil
import sys
from pathlib import Path

import torch

working_path = Path().cwd().parent          # project root (one level above notebooks/)
sys.path.insert(0, str(working_path))
print(f'Project root: {working_path}')

Project root: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master


In [2]:
MODEL_NAME   = 'linknet_binary_doclaynet_20_percent_seed_7_ce_0.25_dice_1.0_.pth'
DEMO_NAME    = 'binary_text_segmentation'

model_source    = working_path / 'models' / MODEL_NAME
examples_source = working_path / 'examples'
demo_path       = working_path / 'demos' / DEMO_NAME
examples_dest   = demo_path / 'examples'

assert model_source.exists(),    f'Model not found: {model_source}'
assert examples_source.exists(), f'Examples dir not found: {examples_source}'
print('[OK] All source paths verified.')

[OK] All source paths verified.


## 1. Create Demo Folder Structure

In [3]:
# Remove any previous build and start fresh
if demo_path.exists():
    shutil.rmtree(demo_path)
    print(f'[INFO] Removed existing: {demo_path}')

demo_path.mkdir(parents=True, exist_ok=True)
examples_dest.mkdir(parents=True, exist_ok=True)
print(f'[OK] Created: {demo_path}')
print(f'[OK] Created: {examples_dest}')

[OK] Created: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\demos\binary_text_segmentation
[OK] Created: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\demos\binary_text_segmentation\examples


## 2. Copy Example Images

In [4]:
binary_example_files = [
    'binary_1_img.png', 'binary_1_mask.png',
    'binary_2_img.png', 'binary_2_mask.png',
    'binary_3_img.png', 'binary_3_mask.png',
]

for fname in binary_example_files:
    src = examples_source / fname
    assert src.exists(), f'Missing example: {src}'
    shutil.copy(src, examples_dest / fname)
    print(f'  Copied: {fname}')

print(f'[OK] {len(binary_example_files)} example files copied to {examples_dest}')

  Copied: binary_1_img.png
  Copied: binary_1_mask.png
  Copied: binary_2_img.png
  Copied: binary_2_mask.png
  Copied: binary_3_img.png
  Copied: binary_3_mask.png
[OK] 6 example files copied to c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\demos\binary_text_segmentation\examples


## 3. Copy Trained Model

In [5]:
shutil.copy(model_source, demo_path / MODEL_NAME)
model_size_mb = (demo_path / MODEL_NAME).stat().st_size / 1e6
print(f'[OK] Copied model: {MODEL_NAME}  ({model_size_mb:.1f} MB)')

[OK] Copied model: linknet_binary_doclaynet_20_percent_seed_7_ce_0.25_dice_1.0_.pth  (46.3 MB)


## 4. Write `model.py`

The LinkNet architecture is inlined here so the demo folder is fully self-contained — no `src/` package needed on HuggingFace Spaces.

In [6]:
model_py = '''"""
Author: Juan Pablo Triana Martinez
LinkNet architecture — standalone for HuggingFace Spaces deployment.
"""
import torch
import torch.nn as nn


class LinknetStem(nn.Module):
    def __init__(self, m: int = 3, n: int = 64) -> None:
        super().__init__()
        self.linknet_stem = nn.Sequential(
            nn.Conv2d(m, n, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linknet_stem(x)


class LinknetEncoderBlock(nn.Module):
    def __init__(self, m: int, n: int) -> None:
        super().__init__()
        self.convs_blocks_1 = nn.Sequential(
            nn.Conv2d(m, n, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.Conv2d(n, n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
        )
        self.skip_conn = nn.Sequential(
            nn.Conv2d(m, n, kernel_size=(1, 1), stride=(2, 2), padding=(0, 0), bias=False),
            nn.BatchNorm2d(n),
        )
        self.convs_block_2 = nn.Sequential(
            nn.Conv2d(n, n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.Conv2d(n, n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.convs_blocks_1(x)
        x2 = x1 + self.skip_conn(x)
        x3 = self.convs_block_2(x2)
        return x3 + x2


class LinknetDecoderBlock(nn.Module):
    def __init__(self, m: int, n: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(m, m // 4, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(m // 4),
            nn.ReLU(),
        )
        self.upsample_block = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(m // 4, m // 4, kernel_size=(3, 3), padding=(1, 1), bias=False),
            nn.BatchNorm2d(m // 4),
            nn.ReLU(),
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(m // 4, n, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_block_1(x)
        x = self.upsample_block(x)
        return self.conv_block_2(x)


class LinknetReconstructer(nn.Module):
    def __init__(self, N: int = 1, m: int = 64, n: int = 32) -> None:
        super().__init__()
        self.upsample_block_1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(m, n, kernel_size=(3, 3), padding=(1, 1), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
        )
        self.conv_block = nn.Sequential(
            nn.Conv2d(n, n, kernel_size=(3, 3), padding=(1, 1), bias=False),
            nn.BatchNorm2d(n),
            nn.ReLU(),
        )
        self.upsample_block_2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(n, N, kernel_size=(3, 3), padding=(1, 1), bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.upsample_block_1(x)
        x = self.conv_block(x)
        return self.upsample_block_2(x)


class LinknetModel(nn.Module):
    def __init__(self, Cin: int = 3, N: int = 1) -> None:
        super().__init__()
        self.stem            = LinknetStem(m=Cin, n=64)
        self.encoder_block_1 = LinknetEncoderBlock(64, 64)
        self.encoder_block_2 = LinknetEncoderBlock(64, 128)
        self.encoder_block_3 = LinknetEncoderBlock(128, 256)
        self.encoder_block_4 = LinknetEncoderBlock(256, 512)
        self.decoder_block_4 = LinknetDecoderBlock(512, 256)
        self.decoder_block_3 = LinknetDecoderBlock(256, 128)
        self.decoder_block_2 = LinknetDecoderBlock(128, 64)
        self.decoder_block_1 = LinknetDecoderBlock(64, 64)
        self.reconstructer   = LinknetReconstructer(N=N, m=64, n=32)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x  = self.stem(x)
        x1 = self.encoder_block_1(x)
        x2 = self.encoder_block_2(x1)
        x3 = self.encoder_block_3(x2)
        x4 = self.encoder_block_4(x3)
        x  = self.decoder_block_4(x4) + x3
        x  = self.decoder_block_3(x)  + x2
        x  = self.decoder_block_2(x)  + x1
        x  = self.decoder_block_1(x)
        return self.reconstructer(x)


def create_binary_model() -> LinknetModel:
    """Factory: LinkNet with 3-channel input and 1 binary output channel."""
    return LinknetModel(Cin=3, N=1)
'''

(demo_path / 'model.py').write_text(model_py, encoding='utf-8')
print('[OK] model.py written')

[OK] model.py written


## 5. Verify `model.py` — Quick Import Test

In [7]:
import importlib.util, sys

spec = importlib.util.spec_from_file_location('model', demo_path / 'model.py')
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)

test_model = model_module.create_binary_model()
dummy      = torch.zeros(1, 3, 512, 512)
out        = test_model(dummy)
print(f'[OK] model.py import OK  |  output shape: {out.shape}')   # expect (1, 1, 512, 512)

[OK] model.py import OK  |  output shape: torch.Size([1, 1, 512, 512])


## 6. Write `app.py`

`app.py` is the entry-point for HuggingFace Spaces.  It imports `model.py` locally and contains all inference and Gradio logic.

In [8]:
app_py = '''"""
Author: Juan Pablo Triana Martinez
Gradio app — LinkNet Binary Text Segmentation (DocLayNet).
Entry-point for HuggingFace Spaces.
"""
import gradio as gr
import numpy as np
import torch
from PIL import Image
from torchvision import transforms

from model import create_binary_model

# ── Constants ──────────────────────────────────────────────────────────────────
MEAN       = [0.9329, 0.9343, 0.9341]
STD        = [0.1651, 0.1592, 0.1623]
IMG_SIZE   = 512
THRESHOLD  = 0.5
MODEL_PATH = "linknet_binary_doclaynet_20_percent_seed_7_ce_0.25_dice_1.0_.pth"

# ── Transforms ─────────────────────────────────────────────────────────────────
inference_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

api_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ── Model ──────────────────────────────────────────────────────────────────────
device = "cpu"
model  = create_binary_model().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# ── Helpers ────────────────────────────────────────────────────────────────────
def _logits_to_binary_mask(logits: torch.Tensor, threshold: float = 0.5) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    return (probs > threshold).float()


def _overlay_mask(image: Image.Image, mask: torch.Tensor) -> np.ndarray:
    """Green-channel overlay of binary mask on the original image."""
    img_np  = api_transform(image).permute(1, 2, 0).numpy()  # (H, W, 3) in [0,1]
    mask_np = mask.squeeze().cpu().numpy()                    # (H, W) in {0,1}
    overlay = img_np.copy()
    overlay[:, :, 1] = np.maximum(overlay[:, :, 1], mask_np)
    return (overlay * 255).clip(0, 255).astype(np.uint8)


# ── Inference ──────────────────────────────────────────────────────────────────
def inference(image: Image.Image, gt_mask: Image.Image = None):
    img_tensor = inference_transform(image).unsqueeze(0)  # (1, 3, H, W)

    with torch.inference_mode():
        logits    = model(img_tensor)                         # (1, 1, H, W)
        pred_mask = _logits_to_binary_mask(logits, THRESHOLD) # (1, 1, H, W)

    pred_mask_np = pred_mask.squeeze().cpu().numpy()  # (H, W) for Gradio display
    overlay      = _overlay_mask(image, pred_mask)

    if gt_mask is None:
        return pred_mask_np, overlay, "No ground truth provided \u2192 metrics unavailable"

    gt_tensor = api_transform(gt_mask.convert("L")).unsqueeze(0)  # (1, 1, H, W)
    gt_binary = (gt_tensor > 0.5).float()

    inter = (pred_mask * gt_binary).sum()
    union = ((pred_mask + gt_binary) > 0).float().sum()
    iou   = (inter / (union  + 1e-7)).item()
    dice  = (2 * inter / (pred_mask.sum() + gt_binary.sum() + 1e-7)).item()

    metrics_text = f"IoU  (Jaccard): {iou:.4f}\nDice (F1):      {dice:.4f}"
    return pred_mask_np, overlay, metrics_text


# ── Gradio Interface ───────────────────────────────────────────────────────────
demo = gr.Interface(
    fn=inference,
    inputs=[
        gr.Image(type="pil", label="Input Image"),
        gr.Image(type="pil", label="Ground Truth Mask (optional)"),
    ],
    outputs=[
        gr.Image(label="Predicted Binary Mask"),
        gr.Image(label="Overlay"),
        gr.Textbox(label="Metrics"),
    ],
    title="LinkNet Binary Text Segmentation \u2014 DocLayNet ChestNut \U0001f330",
    description=(
        "Upload a document image to obtain its binary text segmentation mask. "
        "Optionally upload a ground-truth binary mask to compute IoU and Dice metrics."
    ),
    examples=[
        ["examples/binary_1_img.png", "examples/binary_1_mask.png"],
        ["examples/binary_2_img.png", "examples/binary_2_mask.png"],
        ["examples/binary_3_img.png", "examples/binary_3_mask.png"],
    ],
)

if __name__ == "__main__":
    demo.launch()
'''

(demo_path / 'app.py').write_text(app_py, encoding='utf-8')
print('[OK] app.py written')

[OK] app.py written


## 7. Write `requirements.txt`

In [9]:
requirements_txt = 'torch>=2.2\ntorchvision>=0.17\ngradio\nnumpy>=1.26\n'

(demo_path / 'requirements.txt').write_text(requirements_txt, encoding='utf-8')
print('[OK] requirements.txt written')
print(requirements_txt)

[OK] requirements.txt written
torch>=2.2
torchvision>=0.17
gradio
numpy>=1.26



## 8. Verify Folder Structure

In [10]:
print(f'\nContents of {demo_path}:\n')
for p in sorted(demo_path.rglob('*')):
    indent = '    ' * (len(p.relative_to(demo_path).parts) - 1)
    size   = f'  ({p.stat().st_size / 1e6:.1f} MB)' if p.is_file() else ''
    print(f'{indent}{p.name}{size}')


Contents of c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\demos\binary_text_segmentation:

__pycache__
    model.cpython-312.pyc  (0.0 MB)
app.py  (0.0 MB)
examples
    binary_1_img.png  (0.3 MB)
    binary_1_mask.png  (0.0 MB)
    binary_2_img.png  (0.1 MB)
    binary_2_mask.png  (0.0 MB)
    binary_3_img.png  (0.1 MB)
    binary_3_mask.png  (0.0 MB)
linknet_binary_doclaynet_20_percent_seed_7_ce_0.25_dice_1.0_.pth  (46.3 MB)
model.py  (0.0 MB)
requirements.txt  (0.0 MB)


## 9. Test the App Locally (Optional)

Run the cell below to launch a live Gradio demo directly from this notebook.  
**Skip if you just want to zip and upload.**

In [12]:
import os
os.chdir(demo_path)          # run app from inside the demo folder so relative imports work

import importlib, sys
if 'app' in sys.modules:
    del sys.modules['app']
if 'model' in sys.modules:
    del sys.modules['model']

import app as demo_app
demo_app.demo.launch(debug=False, share=True, allowed_paths=[str(demo_path / 'examples')])

os.chdir(working_path / 'notebooks')   # restore working directory

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://cff588374fee6d2158.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Zip for HuggingFace Upload

The zip file will be placed at `demos/binary_text_segmentation.zip`.  
Upload the zip contents to a new HuggingFace Space (SDK: Gradio).

In [13]:
zip_output = working_path / 'demos' / DEMO_NAME
shutil.make_archive(
    base_name=str(zip_output),
    format='zip',
    root_dir=str(demo_path),
    base_dir='.'
)
zip_path = zip_output.with_suffix('.zip')
zip_size_mb = zip_path.stat().st_size / 1e6
print(f'[OK] Zip created: {zip_path}  ({zip_size_mb:.1f} MB)')

[OK] Zip created: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\demos\binary_text_segmentation.zip  (43.3 MB)


In [14]:
# Download the zip (Google Colab only — skipped silently on local Jupyter)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(f'Not running in Colab.  Find the zip at:\n  {zip_path}')

Not running in Colab.  Find the zip at:
  c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\demos\binary_text_segmentation.zip
